In [3]:
from pyspark.sql import functions as F

StatementMeta(, 51e602e9-81b1-44c6-828e-bb77a012250d, 5, Finished, Available, Finished, False)

In [4]:
df_raw_cust = spark.table("raw_customers")

df_clean_cust = (df_raw_cust
    .filter(F.col("customer_id").isNotNull())          # remove null PKs
    .dropDuplicates(["customer_id"])                    # deduplicate
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("city",          F.trim(F.col("city")))
    .withColumn("segment",       F.trim(F.col("segment")))
)

df_clean_cust.write.mode("overwrite").format("delta") \
    .saveAsTable("clean_customers")

StatementMeta(, 51e602e9-81b1-44c6-828e-bb77a012250d, 6, Finished, Available, Finished, False)

In [6]:
df_raw_prod = spark.table("raw_products")

df_clean_prod = (df_raw_prod
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .filter(F.col("unit_price") > 0)                   # remove invalid prices
)

df_clean_prod.write.mode("overwrite").format("delta") \
    .saveAsTable("clean_products")

StatementMeta(, 51e602e9-81b1-44c6-828e-bb77a012250d, 8, Finished, Available, Finished, False)

In [7]:
df_raw_tx = spark.table("raw_transactions")

df_clean_tx = (df_raw_tx
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["order_id"])
    .withColumn("order_date",   F.to_date(F.col("order_date"), "yyyy-MM-dd"))
    .withColumn("quantity",     F.col("quantity").cast("int"))
    .withColumn("line_amount",  F.col("line_amount").cast("double"))
    .filter(F.col("quantity") > 0)
    .filter(F.col("line_amount") > 0)
)

df_clean_tx.write.mode("overwrite").format("delta") \
    .saveAsTable("clean_transactions")

StatementMeta(, 51e602e9-81b1-44c6-828e-bb77a012250d, 9, Finished, Available, Finished, False)

In [8]:
print(f"  clean_customers:    {df_clean_cust.count()} rows")
print(f"  clean_products:     {df_clean_prod.count()} rows")
print(f"  clean_transactions: {df_clean_tx.count()} rows")

StatementMeta(, 51e602e9-81b1-44c6-828e-bb77a012250d, 10, Finished, Available, Finished, False)

  clean_customers:    50 rows
  clean_products:     30 rows
  clean_transactions: 500 rows


In [11]:
df_dim_customer = (spark.table("clean_customers")
    .select("customer_id", "customer_name", "city", "segment")
    .withColumn("dim_customer_key", F.monotonically_increasing_id())  # surrogate key
)

df_dim_customer.write.mode("overwrite").format("delta") \
    .saveAsTable("DimCustomer")

df_dim_product = (spark.table("clean_products")
    .select("product_id", "product_name", "category", "unit_price")
    .withColumn("dim_product_key", F.monotonically_increasing_id())
)

df_dim_product.write.mode("overwrite").format("delta") \
    .saveAsTable("DimProduct")


df_tx   = spark.table("clean_transactions")
df_cust = spark.table("DimCustomer").select("customer_id", "dim_customer_key")
df_prod = spark.table("DimProduct").select("product_id", "dim_product_key")


df_fact = (df_tx
    .join(df_cust, on="customer_id", how="left")
    .join(df_prod, on="product_id",  how="left")
    .withColumn("year",  F.year("order_date"))
    .withColumn("month", F.month("order_date"))
    .withColumn("month_label", F.date_format("order_date", "yyyy-MM"))
    .select(
        "order_id",
        "order_date", "year", "month", "month_label",
        "dim_customer_key", "customer_id",
        "dim_product_key",  "product_id",
        "quantity",
        F.col("line_amount").alias("revenue")
    )
)

df_fact.write.mode("overwrite").format("delta") \
    .saveAsTable("FactSales")

print(f"  DimCustomer: {df_dim_customer.count()} rows")
print(f"  DimProduct:  {df_dim_product.count()} rows")
print(f"  FactSales:   {df_fact.count()} rows")

StatementMeta(, 51e602e9-81b1-44c6-828e-bb77a012250d, 13, Finished, Available, Finished, False)

  DimCustomer: 50 rows
  DimProduct:  30 rows
  FactSales:   500 rows
